# Extract and Consolidate Tables from Word Document

This notebook extracts tables from the Word document and consolidates them into a single CSV file.


In [ ]:
!uv pip install python-docx


Using Python 3.11.14 environment at: /home/abhishek/Documents/heatpump_ai_2/climate_data_prep/.venv
Resolved 3 packages in 109ms                                         
Installed 1 package in 23ms                                 
 + python-docx==1.2.0


In [1]:
from docx import Document
import csv
import os
import pandas as pd
from pathlib import Path


In [2]:
import os
import csv
from docx import Document


def is_category_row(cells):
    non_empty = [c for c in cells if c]
    return len(non_empty) >= 1 and len(set(non_empty)) == 1


def extract_tables_to_csv(docx_path, output_dir):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    document = Document(docx_path)

    for table_index, table in enumerate(document.tables, start=1):
        csv_path = os.path.join(output_dir, f"table_{table_index}.csv")

        current_category = None
        header_written = False

        with open(csv_path, mode="w", newline="", encoding="utf-8") as csv_file:
            writer = csv.writer(csv_file)

            for row in table.rows:
                cells = [
                    cell.text.replace("\n", " ").strip()
                    for cell in row.cells
                ]

                # ✅ FINAL category detection
                if is_category_row(cells):
                    current_category = cells[0]
                    continue

                # Skip fully empty rows
                if not any(cells):
                    continue

                # Write header once
                if not header_written:
                    writer.writerow(["Kategorie"] + cells)
                    header_written = True
                    continue

                # Write data row
                writer.writerow([current_category] + cells)

        print(f"Saved: {csv_path}")


In [3]:
# Configuration
DOCX_FILE = "beg_waermepumpen_pruef_effizienznachweis.docx"
OUTPUT_BASE_DIR = "/mnt/d/heatpump_data/heatpump_table"
OUTPUT_DIR = os.path.join(OUTPUT_BASE_DIR, "tables_csv")
CONSOLIDATED_CSV = os.path.join(OUTPUT_BASE_DIR, "consolidated_table_1.csv")

# Create output directory if it doesn't exist
os.makedirs(OUTPUT_BASE_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Extract tables to separate CSVs
extract_tables_to_csv(DOCX_FILE, OUTPUT_DIR)


Saved: /mnt/d/heatpump_data/heatpump_table/tables_csv/table_1.csv
Saved: /mnt/d/heatpump_data/heatpump_table/tables_csv/table_2.csv
Saved: /mnt/d/heatpump_data/heatpump_table/tables_csv/table_3.csv
Saved: /mnt/d/heatpump_data/heatpump_table/tables_csv/table_4.csv
Saved: /mnt/d/heatpump_data/heatpump_table/tables_csv/table_5.csv
Saved: /mnt/d/heatpump_data/heatpump_table/tables_csv/table_6.csv
Saved: /mnt/d/heatpump_data/heatpump_table/tables_csv/table_7.csv
Saved: /mnt/d/heatpump_data/heatpump_table/tables_csv/table_8.csv
Saved: /mnt/d/heatpump_data/heatpump_table/tables_csv/table_9.csv
Saved: /mnt/d/heatpump_data/heatpump_table/tables_csv/table_10.csv
Saved: /mnt/d/heatpump_data/heatpump_table/tables_csv/table_11.csv
Saved: /mnt/d/heatpump_data/heatpump_table/tables_csv/table_12.csv
Saved: /mnt/d/heatpump_data/heatpump_table/tables_csv/table_13.csv
Saved: /mnt/d/heatpump_data/heatpump_table/tables_csv/table_14.csv
Saved: /mnt/d/heatpump_data/heatpump_table/tables_csv/table_15.csv
Save

In [5]:
import os

for i in (1, 2):
    path = os.path.join(OUTPUT_DIR, f"table_{i}.csv")
    if os.path.exists(path):
        os.remove(path)

In [7]:


# Translation mapping from German to English headers
header_translation = {
    'Kategorie': 'Category',
    'Hersteller': 'Manufacturer',
    'Typ': 'Type',
    'Niedertemperatur- Anwendung 35 °C': 'Low Temperature Application 35 C',
    'Niedertemperatur- Anwendung 35 °C.1': 'Low Temperature Application 35 C COP',
    'Niedertemperatur- Anwendung 55 °C': 'Low Temperature Application 55 C',
    'Niedertemperatur- Anwendung 55 °C.1': 'Low Temperature Application 55 C COP',
    'Kältemittel': 'Refrigerant',
    'Verfügbarkeit (Siehe Hinweis auf Seite 5)': 'Availability',
    'Verfügbarkeit (Siehe Hinweis auf Seite 5).1': 'Availability (2)'
}

def translate_header(german_header):
    if german_header in header_translation:
        return header_translation[german_header]
    for de, en in header_translation.items():
        if de in german_header or german_header in de:
            return en
    return german_header


csv_files = sorted(Path(OUTPUT_DIR).glob("table_*.csv"))

if not csv_files:
    print("No CSV files found!")
else:
    print(f"Found {len(csv_files)} CSV files to consolidate")

    # --- First pass: determine max column count and reference header ---
    max_cols = 0
    reference_header_german = None

    for csv_file in csv_files:
        df = pd.read_csv(csv_file, encoding="utf-8")
        max_cols = max(max_cols, len(df.columns))

    # Expecting 10 columns now (Kategorie + 9 data columns)
    for csv_file in csv_files:
        df = pd.read_csv(csv_file, encoding="utf-8")
        if len(df.columns) == max_cols:
            reference_header_german = df.columns.tolist()
            print(f"Using reference header from {csv_file.name}")
            break

    reference_header = [translate_header(h) for h in reference_header_german]

    print(f"Maximum columns found: {max_cols}")
    print(f"Reference header (German): {reference_header_german}")
    print(f"Reference header (English): {reference_header}\n")

    # --- Second pass: consolidate ---
    dataframes = []

    for csv_file in csv_files:
        df = pd.read_csv(csv_file, encoding="utf-8")
        original_num_cols = len(df.columns)
        
        # Translate headers first
        translated_headers = [translate_header(h) for h in df.columns]
        
        # Create a mapping from translated headers to their positions in reference_header
        # This ensures columns are placed in the correct positions
        column_mapping = {}
        for i, col in enumerate(translated_headers):
            if col in reference_header:
                column_mapping[col] = i
        
        # Create new dataframe with correct column order and missing columns filled
        new_data = {}
        num_rows = len(df)
        for ref_col in reference_header:
            if ref_col in column_mapping:
                # Column exists, use it
                new_data[ref_col] = df.iloc[:, column_mapping[ref_col]].values
            else:
                # Column is missing, add empty column with correct length
                new_data[ref_col] = [""] * num_rows
        
        df_aligned = pd.DataFrame(new_data)
        df_aligned.columns = reference_header

        # --- Header row detection (skip Category column!) ---
        header_values = [str(x).strip().lower() for x in reference_header[1:]]

        rows_to_drop = []
        for idx in range(min(2, len(df_aligned))):
            row_values = [
                str(x).strip().lower()
                for x in df_aligned.iloc[idx].tolist()[1:]  # skip Category
            ]

            keyword_hits = sum(
                1 for v in row_values
                if any(k in v for k in ['hersteller', 'typ', 'wärme', 'etas', 'refrigerant',
                                        'availability', 'manufacturer', 'type', 'heat'])
            )

            if keyword_hits >= 3:
                rows_to_drop.append(df_aligned.index[idx])

        if rows_to_drop:
            df_aligned = df_aligned.drop(index=rows_to_drop).reset_index(drop=True)

        if len(df_aligned) > 0:
            dataframes.append(df_aligned)

    # --- Final merge ---
    if dataframes:
        consolidated_df = pd.concat(dataframes, ignore_index=True)

        # Define final header format matching the image
        # Order: Manufacturer, Type, heat_output_35_C_in_kw, effeciency_35_C, 
        #        heat_output_55_C_in_kw, effeciency_55_C, Refrigerant, 
        #        grid_service_availability, ee_display_availibility, category
        final_header_map = {
            'Manufacturer': 'Manufacturer',
            'Type': 'Type',
            'Low Temperature Application 35 C': 'heat_output_35_C_in_kw',
            'Low Temperature Application 35 C COP': 'effeciency_35_C',
            'Low Temperature Application 55 C': 'heat_output_55_C_in_kw',
            'Low Temperature Application 55 C COP': 'effeciency_55_C',
            'Refrigerant': 'Refrigerant',
            'Availability': 'grid_service_availability',
            'Availability (2)': 'ee_display_availibility',
            'Category': 'category'
        }
        
        # Rename columns to match the desired format
        consolidated_df = consolidated_df.rename(columns=final_header_map)
        
        # Reorder columns to match image: Manufacturer, Type, heat_output_35_C_in_kw, 
        # effeciency_35_C, heat_output_55_C_in_kw, effeciency_55_C, Refrigerant,
        # grid_service_availability, ee_display_availibility, category
        final_column_order = [
            'Manufacturer', 'Type', 'heat_output_35_C_in_kw', 'effeciency_35_C',
            'heat_output_55_C_in_kw', 'effeciency_55_C', 'Refrigerant',
            'grid_service_availability', 'ee_display_availibility', 'category'
        ]
        consolidated_df = consolidated_df[final_column_order]

        consolidated_df.to_csv(CONSOLIDATED_CSV, index=False, encoding="utf-8")

        print(f"\nConsolidated table saved to: {CONSOLIDATED_CSV}")
        print(f"Total rows: {len(consolidated_df)}")
        print(f"Total columns: {len(consolidated_df.columns)}")
        print(f"Column names: {list(consolidated_df.columns)}")
    else:
        print("No tables could be consolidated!")


Found 441 CSV files to consolidate
Using reference header from table_100.csv
Maximum columns found: 10
Reference header (German): ['Kategorie', 'Hersteller', 'Typ', 'Niedertemperatur- Anwendung 35 °C', 'Niedertemperatur- Anwendung 35 °C.1', 'Niedertemperatur- Anwendung 55 °C', 'Niedertemperatur- Anwendung 55 °C.1', 'Kältemittel', 'Verfügbarkeit (Siehe Hinweis auf Seite 5)', 'Verfügbarkeit (Siehe Hinweis auf Seite 5).1']
Reference header (English): ['Category', 'Manufacturer', 'Type', 'Low Temperature Application 35 C', 'Low Temperature Application 35 C COP', 'Low Temperature Application 55 C', 'Low Temperature Application 55 C COP', 'Refrigerant', 'Availability', 'Availability (2)']


Consolidated table saved to: /mnt/d/heatpump_data/heatpump_table/consolidated_table_1.csv
Total rows: 11994
Total columns: 10
Column names: ['Manufacturer', 'Type', 'heat_output_35_C_in_kw', 'effeciency_35_C', 'heat_output_55_C_in_kw', 'effeciency_55_C', 'Refrigerant', 'grid_service_availability', 'ee_dis

In [ ]:
# # Load consolidated CSV and remove category header rows
# consolidated_df = pd.read_csv(CONSOLIDATED_CSV, encoding="utf-8")

# print(f"Original rows: {len(consolidated_df)}")

# # Identify rows that are likely category headers (e.g., "Luft / Wasser", "Abluft / Wasser", etc.)
# # These are rows where most columns are empty and the first column contains category names
# category_patterns = ['Luft / Wasser', 'Luft/Wasser', 'Abluft / Wasser', 'Luft / Luft', 
#                     'Direktverdampfung / Wasser', 'Erdreich / Wasser', 'Sole / Wasser', 'Wasser / Wasser']

# # Second header row pattern: "Wärme- Nennleistung KW", "ETAs 35 %", "ETAs 55 %", "Netzdien- lichkeit", "EE-Anzeige"
# second_header_keywords = ['Wärme- Nennleistung', 'Nennleistung', 'ETAs', 'Netzdien', 'EE-Anzeige', 'KW', '%']

# # Drop rows where:
# # 1. First column contains a category pattern, OR
# # 2. First column is non-empty but most other columns are empty (likely a header row), OR
# # 3. Row contains second header row pattern (Wärme- Nennleistung, ETAs, Netzdien, EE-Anzeige)
# rows_to_drop = []

# for idx, row in consolidated_df.iterrows():
#     first_col = str(row.iloc[0]).strip()
#     row_text = ' '.join([str(x).strip() for x in row.values]).lower()
    
#     # Check if first column matches a category pattern
#     if any(pattern in first_col for pattern in category_patterns):
#         rows_to_drop.append(idx)
#     # Check for second header row pattern
#     elif any(keyword.lower() in row_text for keyword in second_header_keywords):
#         # Count how many second header keywords are present
#         keyword_matches = sum(1 for keyword in second_header_keywords if keyword.lower() in row_text)
#         if keyword_matches >= 3:  # If 3+ keywords found, likely the second header row
#             rows_to_drop.append(idx)
#     # Check if first column is non-empty but most other columns (2-9) are empty
#     elif first_col and first_col != 'nan':
#         non_empty_count = sum(1 for i in range(1, len(row)) if str(row.iloc[i]).strip() and str(row.iloc[i]).strip() != 'nan')
#         if non_empty_count <= 1:  # If only 0-1 other columns have data, likely a header
#             rows_to_drop.append(idx)

# # Drop the identified rows
# consolidated_df_cleaned = consolidated_df.drop(index=rows_to_drop).reset_index(drop=True)

# print(f"Rows dropped: {len(rows_to_drop)}")
# print(f"Remaining rows: {len(consolidated_df_cleaned)}")

# # Save cleaned consolidated CSV
# consolidated_df_cleaned.to_csv(CONSOLIDATED_CSV, index=False, encoding="utf-8")
# print(f"\nCleaned consolidated table saved to: {CONSOLIDATED_CSV}")


Original rows: 11998
Rows dropped: 5
Remaining rows: 11993

Cleaned consolidated table saved to: /mnt/d/heatpump_data/heatpump_table/consolidated_table_1.csv


In [8]:
consolidated_df_cleaned = consolidated_df.copy()

In [9]:
# Display first few rows of consolidated table
consolidated_df_cleaned.head(n = 30)


,Manufacturer,Type,heat_output_35_C_in_kw,effeciency_35_C,heat_output_55_C_in_kw,effeciency_55_C,Refrigerant,grid_service_availability,ee_display_availibility,category
0,AERMEC GmbH,SKG 250 / SKG250W,"2,7","181,0",NaN,,R32,optional,nein,Luft / Luft (Heizleistung <= 12 kW)
1,Bosch Thermotechnik GmbH,CL7000i-Set 20 E - CL7000iU W 20 E / CL7000i 20 E,"2,3","201,0",NaN,,R32,optional,optional,Luft / Luft (Heizleistung <= 12 kW)
2,Bosch Thermotechnik GmbH,CL7000i-Set 26 E - CL7000iU W 26 E / CL7000i 26 E,"4,1","201,0",NaN,,R32,optional,optional,Luft / Luft (Heizleistung <= 12 kW)
3,Bosch Thermotechnik GmbH,CL7000i-Set 26 EB - CL7000iU W 26 EB / CL7000i...,"4,1","201,0",NaN,,R32,optional,optional,Luft / Luft (Heizleistung <= 12 kW)
4,Bosch Thermotechnik GmbH,CL7000i-Set 26 ES - CL7000iU W 26 ES / CL7000i...,"4,1","201,0",NaN,,R32,optional,optional,Luft / Luft (Heizleistung <= 12 kW)
5,Bosch Thermotechnik GmbH,CL7000i-Set 35 E - CL7000iU W 35 E / CL7000i 35 E,"4,1","201,0",NaN,,R32,optional,optional,Luft / Luft (Heizleistung <= 12 kW)
6,Bosch Thermotechnik GmbH,CL7000i-Set 35 EB - CL7000iU W 35 EB / CL7000i...,"4,1","201,0",NaN,,R32,optional,optional,Luft / Luft (Heizleistung <= 12 kW)
7,Bosch Thermotechnik GmbH,CL7000i-Set 35 ES - CL7000iU W 35 ES / CL7000i...,"4,1","201,0",NaN,,R32,optional,optional,Luft / Luft (Heizleistung <= 12 kW)
8,Bosch Thermotechnik GmbH,CL7000i-Set 53 E - CL7000iU W 53 E / CL7000i 53 E,"5,6","181,0",NaN,,R32,optional,optional,Luft / Luft (Heizleistung <= 12 kW)
9,Bosch Thermotechnik GmbH,CL7000M 53/2 E,"5,3","181,0",NaN,,R32,optional,optional,Luft / Luft (Heizleistung <= 12 kW)


In [32]:
# Display basic info about the consolidated table
consolidated_df_cleaned.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11995 entries, 0 to 11994
Data columns (total 10 columns):
 #   Column                                Non-Null Count  Dtype 
---  ------                                --------------  ----- 
 0   Category                              11995 non-null  object
 1   Manufacturer                          11995 non-null  object
 2   Type                                  11995 non-null  object
 3   Low Temperature Application 35 C      11753 non-null  object
 4   Low Temperature Application 35 C COP  11807 non-null  object
 5   Low Temperature Application 55 C      8911 non-null   object
 6   Low Temperature Application 55 C COP  11110 non-null  object
 7   Refrigerant                           11846 non-null  object
 8   Availability                          11993 non-null  object
 9   Availability (2)                      11993 non-null  object
dtypes: object(10)
memory usage: 937.2+ KB


In [33]:
df = consolidated_df_cleaned.copy()

In [34]:
df.head(n = 30)

,Category,Manufacturer,Type,Low Temperature Application 35 C,Low Temperature Application 35 C COP,Low Temperature Application 55 C,Low Temperature Application 55 C COP,Refrigerant,Availability,Availability (2)
0,,,,Heat Output KW,Efficiency 35 %,Heat Output KW,Efficiency 55 %,,Grid Service,EE Display
1,Luft / Luft (Heizleistung <= 12 kW),AERMEC GmbH,SKG 250 / SKG250W,"2,7","181,0",NaN,R32,optional,nein,
2,Luft / Luft (Heizleistung <= 12 kW),Bosch Thermotechnik GmbH,CL7000i-Set 20 E - CL7000iU W 20 E / CL7000i 20 E,"2,3","201,0",NaN,R32,optional,optional,
3,Luft / Luft (Heizleistung <= 12 kW),Bosch Thermotechnik GmbH,CL7000i-Set 26 E - CL7000iU W 26 E / CL7000i 26 E,"4,1","201,0",NaN,R32,optional,optional,
4,Luft / Luft (Heizleistung <= 12 kW),Bosch Thermotechnik GmbH,CL7000i-Set 26 EB - CL7000iU W 26 EB / CL7000i...,"4,1","201,0",NaN,R32,optional,optional,
5,Luft / Luft (Heizleistung <= 12 kW),Bosch Thermotechnik GmbH,CL7000i-Set 26 ES - CL7000iU W 26 ES / CL7000i...,"4,1","201,0",NaN,R32,optional,optional,
6,Luft / Luft (Heizleistung <= 12 kW),Bosch Thermotechnik GmbH,CL7000i-Set 35 E - CL7000iU W 35 E / CL7000i 35 E,"4,1","201,0",NaN,R32,optional,optional,
7,Luft / Luft (Heizleistung <= 12 kW),Bosch Thermotechnik GmbH,CL7000i-Set 35 EB - CL7000iU W 35 EB / CL7000i...,"4,1","201,0",NaN,R32,optional,optional,
8,Luft / Luft (Heizleistung <= 12 kW),Bosch Thermotechnik GmbH,CL7000i-Set 35 ES - CL7000iU W 35 ES / CL7000i...,"4,1","201,0",NaN,R32,optional,optional,
9,Luft / Luft (Heizleistung <= 12 kW),Bosch Thermotechnik GmbH,CL7000i-Set 53 E - CL7000iU W 53 E / CL7000i 53 E,"5,6","181,0",NaN,R32,optional,optional,


In [ ]:
# # --- Add Kategorie column based on section rows ---
# import pandas as pd

# def is_section_row(row):
#     non_empty = [str(c).strip() for c in row if pd.notna(c) and str(c).strip() != ""]
#     return len(non_empty) == 1

# current_category = None
# records = []

# for _, row in df.iterrows():
#     values = [str(c).strip() if pd.notna(c) else "" for c in row]

#     if is_section_row(values):
#         current_category = values[0]
#         continue

#     if not values[0]:
#         continue

#     record = {"Kategorie": current_category}
#     record.update(dict(zip(df.columns, values)))
#     records.append(record)

# df = pd.DataFrame(records)

# # Optional sanity check
# assert df['Category'].notna().all()

# df.to_csv("consolidated_table_with_category.csv", index=False)


KeyError: 'Kategorie'

In [40]:
!uv pip install openpyxl

Using Python 3.11.14 environment at: /home/abhishek/Documents/heatpump_ai_2/climate_data_prep/.venv
Resolved 2 packages in 149ms                                         
Installed 2 packages in 52ms                                
 + et-xmlfile==2.0.0
 + openpyxl==3.1.5


In [45]:
import pandas as pd

df1 = pd.read_csv(CONSOLIDATED_CSV)

# df1 = pd.read_csv("consolidated_table_1.csv")

# Drop first row and reset index
df1 = df1.iloc[1:].reset_index(drop=True)
df2 = pd.read_excel("/mnt/c/Users/Abhishek/Downloads/head_load.xlsx")

# Example column alignment (adjust as needed)

rename_map = {
    "Manufacturer": "manufacturer",
    "Type": "model_type",
    "Low Temperature Application 35 C": "heat_output_35_kw",
    "Low Temperature Application 35 C COP": "cop_35",
    "Low Temperature Application 55 C": "heat_output_55_kw",
    "Low Temperature Application 55 C COP": "cop_55",
    "Category": "category",
    "Refrigerant": "refrigerant",
    "Availability": "grid_service_availability",
    "Availability (2)": "ee_display_availability"
}

df1 = df1.rename(columns=rename_map)


# Sort and reset index
df1 = df1.sort_values(list(df1.columns)).reset_index(drop=True)
df2 = df2.sort_values(list(df2.columns)).reset_index(drop=True)

# Exact comparison
df1.equals(df2)


False

In [49]:
df1.head()


,manufacturer,model_type,heat_output_35_kw,cop_35,heat_output_55_kw,cop_55,refrigerant,grid_service_availability,ee_display_availability,category
0,1A HEIZEN STROBL UG,1A COP TOP 14R,"14,0","161,2","8,8","125,0",R32,ja,ja,Direktkondensation im Pufferspeicher (Sonderba...
1,1SINQ GmbH,1SINQ ONE-10,"10,2","220,0","10,0","165,0",R290,ja,ja,Luft / Wasser
2,1SINQ GmbH,1SINQ ONE-14E,"15,4","194,0","14,2","152,0",R290,ja,ja,Luft / Wasser
3,1SINQ GmbH,1SINQ ONE-16,"15,5","217,0","16,0","158,0",R290,ja,ja,Luft / Wasser
4,1SINQ GmbH,1SINQ ONE-18,"18,8","200,0","18,0","158,0",R290,ja,ja,Luft / Wasser


In [51]:
df2.head()

,manufacturer,model_type,heat_output_35_kw,cop_35,heat_output_55_kw,cop_55,refrigerant,grid_service_availability,ee_display_availability,category
0,1A HEIZEN STROBL UG,1A COP TOP 14R,"14,0","161,2","8,8","125,0",R32,ja,ja,Direktkondensation im Pufferspeicher (Sonderba...
1,1SINQ GmbH,1SINQ ONE-10,"10,2","220,0","10,0","165,0",R290,ja,ja,Luft / Wasser
2,1SINQ GmbH,1SINQ ONE-14E,"15,4","194,0","14,2","152,0",R290,ja,ja,Luft / Wasser
3,1SINQ GmbH,1SINQ ONE-16,"15,5","217,0","16,0","158,0",R290,ja,ja,Luft / Wasser
4,1SINQ GmbH,1SINQ ONE-18,"18,8","200,0","18,0","158,0",R290,ja,ja,Luft / Wasser


In [52]:
df1 = df1.reset_index(drop=True)
df2 = df2.reset_index(drop=True)
diff = df1.compare(df2)
diff.head()


ValueError: Can only compare identically-labeled (both index and columns) DataFrame objects

In [53]:
# 1. Ensure same column order
df1 = df1[df2.columns]

# 2. Sort deterministically
sort_cols = list(df1.columns)
df1 = df1.sort_values(sort_cols)
df2 = df2.sort_values(sort_cols)

# 3. Reset index (CRITICAL)
df1 = df1.reset_index(drop=True)
df2 = df2.reset_index(drop=True)

# 4. Compare
df1.equals(df2)      # True / False
diff = df1.compare(df2)

ValueError: Can only compare identically-labeled (both index and columns) DataFrame objects